In [112]:
#The following file paths are all absolute paths. You can replace them with relative paths at runtime, and the files are located in their respective folders.
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import gym
import matplotlib.pyplot as plt
import random
import argparse
from collections import OrderedDict
from copy import copy
# import Learn_Knonlinear as lka
import scipy
import scipy.linalg
from scipy.integrate import odeint
import sys
import os
import csv
sys.path.append("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/Predict_Model_Train/")
sys.path.append("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/utility_LSPN/")
from Utility import data_collecter
from LSPN_test import LSPN_Mamba
os.environ['KMP_DUPLICATE_LIB_OK'] = "TRUE"

In [113]:
Methods = ["KoopmanDerivative","KoopmanRBF",\
            "KNonlinear","KNonlinearRNN","KoopmanU",\
            "KoopmanNonlinearA","KoopmanNonlinear",\
                "KNonlinearmamba"]
Method_names = ["KoopmanDerivative","KoopmanRBF",\
            "KDNN","KRNN","DKUC(ours)",\
            "DKAC(ours)","DKN(ours)",\
                "KNonlinearmamba"]

In [114]:
def short_predict(method_index,data,net):
    train_traj_num,steps,Nstates = data.shape
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = torch.FloatTensor(data).to(device)
    max_loss_list = []
    mean_loss_list = []
    min_loss_list = []
    pre_list = []
    real_list = []
    if method_index==7:
        X_pred = net.forward(data[:,:steps-1,:])
        # X_pred = X_pred
        # X_pred = X_pred[:,:,u_dim:]
    for i in range(steps-1):
        if method_index==7:     
            X_current = X_pred[:,i,:]
        Y = data[:,i+1,:]
        if method_index==7:
            Err = X_current-Y
            print(torch.mean(X_current,axis=0)[0],torch.mean(Y,axis=0)[0])
            # print(Y.shape)
            pre_list.append(np.array(torch.mean(X_current,axis=0).detach().cpu().numpy()))
            # print(torch.mean(X_current[:Nstate,:].T,axis=0).shape)
            real_list.append(np.array(torch.mean(Y,axis=0).detach().cpu().numpy()))
        else:
            Err = X_current[:2,:].T-Y
            print(torch.mean(X_current[:2,:].T,axis=0)[0],torch.mean(Y,axis=0)[0])
            # print(Y.shape)
            pre_list.append(np.array(torch.mean(X_current[:2,:].T,axis=0).detach().cpu().numpy()))
            # print(torch.mean(X_current[:Nstate,:].T,axis=0).shape)
            real_list.append(np.array(torch.mean(Y,axis=0).detach().cpu().numpy()))
        max_loss_list.append(torch.mean(torch.max(torch.abs(Err),axis=0).values).detach().cpu().numpy())
        mean_loss_list.append(torch.mean(torch.mean(torch.abs(Err),axis=0)).detach().cpu().numpy())
        min_loss_list.append(torch.mean(torch.min(torch.abs(Err),axis=0).values).detach().cpu().numpy())
    print(np.array(pre_list)[:,0], np.array(real_list)[:,0])
    print(np.array(mean_loss_list))
    return np.array(max_loss_list),np.array(mean_loss_list),np.array(min_loss_list),np.array(pre_list),np.array(real_list)

In [115]:
def read_lorenz_dataset_original_shape(file_path, num_samples=100, num_steps=100):
    data = np.zeros((num_samples, num_steps, 3))
    with open(file_path, 'r', newline='') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # 跳过列名行
        for i in range(num_samples):
            for j in range(num_steps):
                row = next(reader)
                data[i, j] = [float(val) for val in row]
    return data

def mean_predict(suffix,env_name,method_index,layer_i,steps):
    # method_index = 0
    method = Methods[method_index]
    root_path = "/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/Mamba_data_raw/"+suffix
    print(method)
    #sys.path.append("control/train/")
    if  method.endswith("KNonlinearRNN"):
        import Learn_Knonlinear_RNN as lka
    elif method.endswith("KNonlinear"):
        import Learn_Knonlinear as lka
    elif method.endswith("KoopmanNonlinear"):
        import learn_DKN_SOC_sizeNN as lka
    elif method.endswith("KoopmanNonlinearA"):
        import learn_DKAC_SOC_sizeNN as lka
    elif method.endswith("KoopmanU"):
        import learn_DKUC_SOC_sizeNN as lka
    elif method.endswith("KNonlinearmamba"):
        import Learn_Knonlinear_mamba_luorenz as lka  
    for file in os.listdir(root_path):
        if file.startswith(method+"_"+"luorenz") and file.endswith(".pth"):
            model_path = file  
    # Data_collect = data_collecter(env_name)
    udim = 1
    Nstates = 2
    layer_depth = layer_i
    layer_width = 128
    dicts = torch.load(root_path+"/"+model_path,map_location=torch.device('cpu'))
    state_dict = dicts["model"]
    if method.endswith("KNonlinear"):
        Elayer = dicts["Elayer"]
        net = lka.Network(layers=Elayer,u_dim=0)
    elif method.endswith("KNonlinearRNN"):
        net = lka.Network(input_size=0+2,output_size=Nstates,hidden_dim=layer_width, n_layers=layer_depth-1)
    elif method.endswith("KNonlinearmamba"):
        net = LSPN_Mamba(
        # This module uses roughly 3 * expand * d_model^2 parameters
        d_model=3, # Model dimension d_model
        d_state=8,  # SSM state expansion factor
        d_conv=4,    # Local convolution width
        expand=4,    # Block expansion factor
    ).to("cuda")
    elif method.endswith("KoopmanNonlinear") or method.endswith("KoopmanNonlinearA"):
        layer = dicts["layer"]
        blayer = dicts["blayer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,blayer,NKoopman,udim)
    elif method.endswith("KoopmanU"):
        layer = dicts["layer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,NKoopman,udim)  
    net.load_state_dict(state_dict)
    #device = torch.device("cpu")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.cuda()
    net.double()
    Samples = 1
    steps = steps
    random.seed(2022)
    np.random.seed(2022)
    times = 1
    m = 50
    pre_list_all = np.zeros((times,steps,Nstates))
    rea_list_all = np.zeros((times,steps,Nstates))
    with torch.no_grad():
        for i in range(times):
            # test_data_path = "D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/"+"method{}{}.npy".format(env_name,i)
            # if os.path.exists(test_data_path):
            #     test_data = np.load("D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/{}{}.npy".format(env_name,i))
            # else:
            X_original_shape = read_lorenz_dataset_original_shape('/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/utility_LSPN/lorenz_data100000.csv',num_steps=100000)
            test_data = X_original_shape[m-Samples:m+1-Samples, :steps, :]
            np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_data/middle_data/predict_future{}{}{}.npy".format(i,suffix,env_name),test_data)
            _,_,_,pre_list,rea_list = short_predict(method_index,test_data,net)
            #print(pre_list.shape)
            pre_list_all = pre_list
            rea_list_all = rea_list
    pre_list_mean = pre_list_all#np.mean(pre_list_all,axis=0)
    rea_list_mean = rea_list_all#np.mean(rea_list_all,axis=0)  
    # np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/"+env_name+"_"+method+"layer1{}{}.npy".format(layer_i, steps),np.array([pre_list_mean, rea_list_mean]))
    np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/"+env_name+"{}-pre-long{}.npy".format(method_index,steps),pre_list_mean)
    np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/"+env_name+"{}-real-long{}.npy".format(method_index,steps),rea_list_mean)
    return pre_list_mean, rea_list_mean

In [116]:
suffix = ["*","*","Knolinear_SOC_models","KRNN_SOC_models","DKUC_SOC_sizeNN","compare_DKAC_sizeNN_30","DKN_SOC_sizeNN","mamba_testluorenz2"]
env_name = "luorenz"

steps = 10000
for i in range(7,8):
    pre_list_mean, rea_list_mean = mean_predict(suffix[i],env_name,method_index=i,layer_i=2,steps=steps)
    print(pre_list_mean.shape)

KNonlinearmamba
tensor(-15.7237, device='cuda:0') tensor(-15.7010, device='cuda:0')
tensor(-15.8127, device='cuda:0') tensor(-15.7929, device='cuda:0')
tensor(-15.9431, device='cuda:0') tensor(-15.9239, device='cuda:0')
tensor(-16.0799, device='cuda:0') tensor(-16.0593, device='cuda:0')
tensor(-16.1845, device='cuda:0') tensor(-16.1691, device='cuda:0')
tensor(-16.2338, device='cuda:0') tensor(-16.2239, device='cuda:0')
tensor(-16.2052, device='cuda:0') tensor(-16.1995, device='cuda:0')
tensor(-16.0820, device='cuda:0') tensor(-16.0785, device='cuda:0')
tensor(-15.8550, device='cuda:0') tensor(-15.8508, device='cuda:0')
tensor(-15.5215, device='cuda:0') tensor(-15.5134, device='cuda:0')
tensor(-15.0775, device='cuda:0') tensor(-15.0701, device='cuda:0')
tensor(-14.5303, device='cuda:0') tensor(-14.5266, device='cuda:0')
tensor(-13.8903, device='cuda:0') tensor(-13.8865, device='cuda:0')
tensor(-13.1703, device='cuda:0') tensor(-13.1671, device='cuda:0')
tensor(-12.3891, device='cuda:0'

In [ ]:
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
font = {'size': 12}
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['font.family'] = ['Times New Roman']
mpl.rcParams["axes.titlepad"] = 16
mpl.rcParams['lines.linewidth'] = 1.5
mpl.rcParams['lines.markersize'] = 3
labels = ["*","*","BaseNet","KoopmanRNN","SDKN-DKUC","SDKN-DKAC","SDKN-DKN"]
plt.figure(figsize=(5, 5))
plt.rc('font', **font)
# markers = ['*','+','*','+','*','+','*']
env_name = env_name
title = env_name
compare = "mean"
T = []
for t in range(steps):
    T.append(t+1)
T = np.array(T)
times = 500
for i in range(1):
    pre_data = pre_list_mean
    real_data = rea_list_mean
    #print(data.shape)
    # plt.plot(T,pre_data[:,0],'--',color = colors[i],label=labels[i])#np.log10(data)
    plt.plot(pre_data[:times,0],pre_data[:times,1],'--',color = colors[i+1],label=labels[i])#np.log10(data)
    # plt.plot(T,real_data[:,0],'-',color = colors[i+2],label=labels[i])#np.log10(data)
    plt.plot(real_data[:times,0],real_data[:times,1],'-',color = colors[i+3],label=labels[i])#np.log10(data)
    plt.legend(loc = 'lower right')
plt.grid(False)
plt.xlabel("steps",fontsize=12)
plt.ylabel("state",fontsize=12)
# plt.ylim([-2,1])
# plt.yticks([])
plt.title(env_name,fontsize=12)
#plt.savefig("D:/毕业设计/论文/pictures/SOC_short_predict/steps{}".format(steps)+env_name+"_models_"+compare+"_new1.png",dpi=500)

In [ ]:
def read_henon_map_dataset_original_shape(file_path):
    data = np.load(file_path)
    return data

def mean_predict(suffix,env_name,method_index,layer_i,steps):
    # method_index = 0
    method = Methods[method_index]
    root_path = "/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/Mamba_data_raw/"+suffix
    print(method)
    #sys.path.append("control/train/")
    if  method.endswith("KNonlinearRNN"):
        import Learn_Knonlinear_RNN as lka
    elif method.endswith("KNonlinear"):
        import Learn_Knonlinear as lka
    elif method.endswith("KoopmanNonlinear"):
        import learn_DKN_SOC_sizeNN as lka
    elif method.endswith("KoopmanNonlinearA"):
        import learn_DKAC_SOC_sizeNN as lka
    elif method.endswith("KoopmanU"):
        import learn_DKUC_SOC_sizeNN as lka
    elif method.endswith("KNonlinearmamba"):
        import Learn_Knonlinear_mamba as lka  
    for file in os.listdir(root_path):
        if file.startswith(method+"_"+"henon_map") and file.endswith(".pth"):
            model_path = file  
    # Data_collect = data_collecter(env_name)
    udim = 1
    Nstates = 2
    layer_depth = layer_i
    layer_width = 128
    dicts = torch.load(root_path+"/"+model_path,map_location=torch.device('cpu'))
    state_dict = dicts["model"]
    if method.endswith("KNonlinear"):
        Elayer = dicts["Elayer"]
        net = lka.Network(layers=Elayer,u_dim=0)
    elif method.endswith("KNonlinearRNN"):
        net = lka.Network(input_size=0+2,output_size=Nstates,hidden_dim=layer_width, n_layers=layer_depth-1)
    elif method.endswith("KNonlinearmamba"):
        net = LSPN_Mamba(
        # This module uses roughly 3 * expand * d_model^2 parameters
        d_model=2, # Model dimension d_model
        d_state=8,  # SSM state expansion factor
        d_conv=4,    # Local convolution width
        expand=4,    # Block expansion factor
    ).to("cuda")
    elif method.endswith("KoopmanNonlinear") or method.endswith("KoopmanNonlinearA"):
        layer = dicts["layer"]
        blayer = dicts["blayer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,blayer,NKoopman,udim)
    elif method.endswith("KoopmanU"):
        layer = dicts["layer"]
        NKoopman = layer[-1]+Nstates
        net = lka.Network(layer,NKoopman,udim)  
    net.load_state_dict(state_dict)
    #device = torch.device("cpu")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.cuda()
    net.double()
    Samples = 1
    steps = steps
    random.seed(2022)
    np.random.seed(2022)
    times = 1
    pre_list_all = np.zeros((times,steps,Nstates))
    rea_list_all = np.zeros((times,steps,Nstates))
    with torch.no_grad():
        for i in range(times):
            # test_data_path = "D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/"+"method{}{}.npy".format(env_name,i)
            # if os.path.exists(test_data_path):
            #     test_data = np.load("D:/毕业设计/中期/Python/MPC_trykoopman/results/SOC_compare_sizeNNdata/{}{}.npy".format(env_name,i))
            # else:
            X_original_shape = read_henon_map_dataset_original_shape('/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/utility_LSPN/henon_map_data_filtered.npy')
            test_data = X_original_shape[-Samples:, :steps, :]
            np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_data/middle_data/predict_future{}{}{}.npy".format(i,suffix,env_name),test_data)
            _,_,_,pre_list,rea_list = short_predict(method_index,test_data,net)
            #print(pre_list.shape)
            pre_list_all = pre_list
            rea_list_all = rea_list
    pre_list_mean = pre_list_all#np.mean(pre_list_all,axis=0)
    rea_list_mean = rea_list_all#np.mean(rea_list_all,axis=0)  
    # np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/"+env_name+"_"+method+"layer1{}{}.npy".format(layer_i, steps),np.array([pre_list_mean, rea_list_mean]))
    np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/"+env_name+"{}-pre-long{}.npy".format(method_index,steps),pre_list_mean)
    np.save("/media/serena/study/Vscode_works/python_Vscode/mamba/Nonlinear_LSPN/DATA/LSPN_predict_future_data/"+env_name+"{}-real-long{}.npy".format(method_index,steps),rea_list_mean)
    return pre_list_mean, rea_list_mean

In [ ]:
suffix = ["*","*","Knolinear_SOC_models","KRNN_SOC_models","DKUC_SOC_sizeNN","compare_DKAC_sizeNN_30","DKN_SOC_sizeNN","mamba_testhenon_map1"]
env_name = "henon_map"
steps = 500
for i in range(7,8):
    pre_list_mean, rea_list_mean = mean_predict(suffix[i],env_name,method_index=i,layer_i=2,steps=steps)
    print(pre_list_mean.shape)

In [ ]:
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
font = {'size': 12}
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['font.family'] = ['Times New Roman']
mpl.rcParams["axes.titlepad"] = 16
mpl.rcParams['lines.linewidth'] = 1.5
mpl.rcParams['lines.markersize'] = 3
labels = ["*","*","BaseNet","KoopmanRNN","SDKN-DKUC","SDKN-DKAC","SDKN-DKN"]
plt.figure(figsize=(5, 5))
plt.rc('font', **font)
# markers = ['*','+','*','+','*','+','*']
env_name = env_name
title = env_name
compare = "mean"
T = []
for t in range(steps):
    T.append(t+1)
T = np.array(T)
times = 200
for i in range(1):
    pre_data = pre_list_mean
    real_data = rea_list_mean
    #print(data.shape)
    # plt.plot(T,pre_data[:,0],'--',color = colors[i],label=labels[i])#np.log10(data)
    plt.plot(pre_data[:times,0],pre_data[:times,1],'--',color = colors[i],label=labels[i])#np.log10(data)
    # plt.plot(T,real_data[:,0],'-',color = colors[i+2],label=labels[i])#np.log10(data)
    plt.plot(real_data[:times,0],real_data[:times,1],'--',color = colors[i+3],label=labels[i])#np.log10(data)
    plt.legend(loc = 'lower right')
plt.grid(False)
plt.xlabel("steps",fontsize=12)
plt.ylabel("state",fontsize=12)
# plt.ylim([-2,1])
# plt.yticks([])
plt.title(env_name,fontsize=12)
#plt.savefig("D:/毕业设计/论文/pictures/SOC_short_predict/steps{}".format(steps)+env_name+"_models_"+compare+"_new1.png",dpi=500)